In [ ]:
import numpy as np
import pandas as pd
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Input, Masking
from tensorflow.keras.optimizers import Adam
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow.keras.backend as K
from imblearn.over_sampling import RandomOverSampler
import gensim.downloader as api
from sklearn.model_selection import train_test_split
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from google.colab import drive
import random
random.seed(42)
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pwd

/content


In [ ]:

raw_data_dir = "/content/drive/MyDrive/CS_224N_Project/Evaluation/Data/Processed_data/Train/"
raw_testdata_dir = "/content/drive/MyDrive/CS_224N_Project/Evaluation/Data/Processed_data/Test/"
file_names = ['abortion.csv', 'before_iraq.csv', 'firearms.csv', 'main_iraq.csv']

def get_data(file_name, data_dir=raw_data_dir):
  data = pd.read_csv(data_dir + file_name, index_col=None)
  np.shape(data)
  return data


def get_test_data(file_name, data_dir=raw_testdata_dir):
  data = pd.read_csv(data_dir + file_name, index_col=None)
  np.shape(data)
  return data

train_abortion_df = get_data('abortion_data.csv')
train_firearms_df = get_data('firearms_data.csv')
train_main_iraq_df = get_data('main_iraq_data.csv')
test_abortion_df = get_test_data('abortion_data.csv')
test_firearms_df = get_test_data('firearms_data.csv')
test_main_iraq_df = get_test_data('main_iraq_data.csv')




train_abortion_label_df = get_data('abortion_labels.csv')['party'].apply(lambda x: 1 if x == 'D' else 0 if x == 'R' else None)
train_firearms_label_df = get_data('firearms_labels.csv')['party'].apply(lambda x: 1 if x == 'D' else 0 if x == 'R' else None)
train_main_iraq_label_df = get_data('main_iraq_labels.csv')['party'].apply(lambda x: 1 if x == 'D' else 0 if x == 'R' else None)
test_abortion_label_df = get_test_data('abortion_labels.csv')['party'].apply(lambda x: 1 if x == 'D' else 0 if x == 'R' else None)
test_firearms_label_df = get_test_data('firearms_labels.csv')['party'].apply(lambda x: 1 if x == 'D' else 0 if x == 'R' else None)
test_main_iraq_label_df = get_test_data('main_iraq_labels.csv')['party'].apply(lambda x: 1 if x == 'D' else 0 if x == 'R' else None)


concat_train_data = get_data('merged_train_data.csv')
concat_val_data = get_data('merged_val_data.csv')
concat_train_label_df = get_data('merged_train_labels.csv')['party'].apply(lambda x: 1 if x == 'D' else 0 if x == 'R' else None)
concat_val_label_df = get_data('merged_val_labels.csv')['party'].apply(lambda x: 1 if x == 'D' else 0 if x == 'R' else None)




from sklearn.model_selection import train_test_split
train_main_iraq, val_main_iraq, train_main_iraq_label, val_main_iraq_label = train_test_split(
    train_main_iraq_df, train_main_iraq_label_df, test_size=0.125, random_state=42
)



                                                  speech        date  \
0      Mr. Chairman. I yield myself such time as I ma...  1999-09-30   
1      Mr. Speaker. this is a day with a lot of shame...  1989-10-24   
2      Mr. Speaker. I rise today in strong support of...  2003-06-04   
3      Madam Chairman. I thank the gentleman for yiel...  1993-03-25   
4      No. Mr. Chairman. What I am suggesting is that...  1996-05-14   
...                                                  ...         ...   
13809  Mr. Speaker. the Iraqi tribunal recently annou...  2006-04-04   
13810  Mr. Speaker. today President Bush. as required...  2007-07-12   
13811  Mr. Speaker. I want to thank Congressman WAMP ...  2007-06-06   
13812  I thank my friend and colleague from the Rules...  2007-07-12   
13813  Mr. Speaker. I am a little bit saddened by the...  2006-06-15   

       word_count state  iraq_count  
0             537    CA         NaN  
1             247    CA         NaN  
2             351    

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def tokenize_texts(train_phrases, test_phrases, num_words=10000):
    tokenizer = Tokenizer(num_words=num_words)
    tokenizer.fit_on_texts(train_phrases)

    tokenized_train = tokenizer.texts_to_sequences(train_phrases)
    #tokenized_validate = tokenizer.texts_to_sequences(validate_phrases)
    tokenized_test = tokenizer.texts_to_sequences(test_phrases)

    max_length = max(len(seq) for seq in tokenized_train + tokenized_test)

    padded_train = pad_sequences(tokenized_train, maxlen=max_length, padding='post')
    #padded_validate = pad_sequences(tokenized_validate, maxlen=max_length, padding='post')
    padded_test = pad_sequences(tokenized_test, maxlen=max_length, padding='post')

    return tokenizer, padded_train, padded_test, max_length

def tokenize_texts_with_validate(train_phrases, validate_phrases, test_phrases, num_words=10000):
    tokenizer = Tokenizer(num_words=num_words)
    tokenizer.fit_on_texts(train_phrases)

    tokenized_train = tokenizer.texts_to_sequences(train_phrases)
    tokenized_validate = tokenizer.texts_to_sequences(validate_phrases)
    tokenized_test = tokenizer.texts_to_sequences(test_phrases)

    max_length = max(len(seq) for seq in tokenized_train + tokenized_validate + tokenized_test)

    padded_train = pad_sequences(tokenized_train, maxlen=max_length, padding='post')
    padded_validate = pad_sequences(tokenized_validate, maxlen=max_length, padding='post')
    padded_test = pad_sequences(tokenized_test, maxlen=max_length, padding='post')

    return tokenizer, padded_train, padded_validate, padded_test, max_length

tokenizer_party, X_train_abortion, X_test_abortion, max_length_party = tokenize_texts(
    train_abortion_df['speech'],
    test_abortion_df['speech']
)

tokenizer_party, X_train_firearm, X_test_firearm, max_length_party = tokenize_texts(
    train_firearms_df['speech'],
    test_firearms_df['speech']
)


tokenize_party, X_train_merge, X_val_merge, max_length_party = tokenize_texts(
    concat_train_data['speech'],
    concat_val_data['speech']
)



tokenizer_party, X_train_main_iraq, X_validate_main_iraq, X_test_main_iraq, max_length_party = tokenize_texts_with_validate(
    train_main_iraq['speech'],
    val_main_iraq['speech'],
    test_main_iraq_df['speech']
)


In [ ]:
## pretraining generla context
word2vec_model = api.load("word2vec-google-news-300")
embedding_dim = word2vec_model.vector_size
num_words = 10000

def create_embedding_matrix(tokenizer, num_words, word2vec_model, embedding_dim):
    embedding_matrix = np.zeros((num_words, embedding_dim))
    for word, i in tokenizer.word_index.items():
        if i < num_words:
            if word in word2vec_model:
                embedding_matrix[i] = word2vec_model[word]
    return embedding_matrix

embedding_matrix_party = create_embedding_matrix(tokenizer_party, num_words, word2vec_model, embedding_dim)

np.save('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/embedding_matrix_party.npy', embedding_matrix_party)

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
import numpy as np

num_words = 10000

# Specify the path to your embedding matrix file
embedding_matrix_file = '/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/embedding_matrix_party.npy'

# Load the embedding matrix
embedding_matrix_party = np.load(embedding_matrix_file)

embedding_dim = embedding_matrix_party.shape[1]


In [ ]:
def build_model(input_shape, embedding_matrix, num_words, embedding_dim, l2_reg, finetune=None):
    text_input = Input(shape=(input_shape,))
    masking = Masking(mask_value=0.0)(text_input)
    embedding = Embedding(input_dim=num_words, output_dim=embedding_dim, weights=[embedding_matrix], trainable=False)(masking)
    bi_lstm1 = Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=l2(l2_reg)))(embedding)
    bi_lstm2 = Bidirectional(LSTM(32, return_sequences=False, kernel_regularizer=l2(l2_reg)))(bi_lstm1)
    dropout = Dropout(0.3)(bi_lstm2)
    output = Dense(1, activation='sigmoid', kernel_regularizer=l2(l2_reg))(dropout)
    model = Model(inputs=text_input, outputs=output)
    return model

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model_party = build_model(max_length_party, embedding_matrix_party, num_words, embedding_dim, 0.0005)
model_party.compile(optimizer=Adam(), loss=BinaryCrossentropy(), metrics=['accuracy'])

def train_and_evaluate(train_data_domain, train_label_domain, train_val_data, train_val_labels, val_data, val_labels, test_data, test_labels, model_party, early_stopping, model_name, finetune_data=None, finetune_labels=None):
    best_weights_filepath = 'best.weights.h5'

    model_checkpoint = ModelCheckpoint(best_weights_filepath, monitor='val_loss', save_best_only=True, save_weights_only=True)

    if finetune_data is not None:
        history = model_party.fit(
            train_data_domain,
            train_label_domain,
            epochs=10,
            batch_size=32,
            validation_data=(train_val_data, train_val_labels),
            callbacks=[early_stopping, model_checkpoint]
        )


        model_party.load_weights(best_weights_filepath)


        history_finetune = model_party.fit(
            finetune_data,
            finetune_labels,
            epochs=10,
            batch_size=32,
            validation_data=(val_data, val_labels),
            callbacks=[early_stopping]
        )

        history = history_finetune

    else:
        history = model_party.fit(
            train_data_domain,
            train_label_domain,
            epochs=10,
            batch_size=32,
            validation_data=(val_data, val_labels),
            callbacks=[early_stopping, model_checkpoint]
        )

    # Evaluate the model on test data
    results = model_party.evaluate(test_data, test_labels)
    return results, model_party, history

In [ ]:

vanilla_results, vanilla_trained_model, training_history = train_and_evaluate(
    X_train_main_iraq, train_main_iraq_label,
    None, None,
    X_validate_main_iraq, val_main_iraq_label,
    X_test_main_iraq, test_main_iraq_label_df,
    model_party, early_stopping, model_name = "vanilla_model"
)

vanilla_trained_model.save('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/model/vanilla_model.h5')


Epoch 1/10
228/228 [==============================] - 97s 237ms/step - loss: 0.7994 - accuracy: 0.5628 - val_loss: 0.6826 - val_accuracy: 0.6465
Epoch 2/10
228/228 [==============================] - 48s 209ms/step - loss: 0.7215 - accuracy: 0.5701 - val_loss: 0.7164 - val_accuracy: 0.6215
Epoch 3/10
228/228 [==============================] - 51s 223ms/step - loss: 0.7143 - accuracy: 0.5384 - val_loss: 0.7123 - val_accuracy: 0.5841
Epoch 4/10
66/66 [==============================] - 6s 95ms/step - loss: 0.6929 - accuracy: 0.6278


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
abortion_learning_results, abortion_learning_trained_model, abortion_learning_history = train_and_evaluate(
    X_train_abortion, train_abortion_label_df,
    X_test_abortion,test_abortion_label_df,
    X_validate_main_iraq, val_main_iraq_label,
    X_test_main_iraq, test_main_iraq_label_df,
    model_party, early_stopping,
    model_name = "abortion_firearm",
    finetune_data=X_train_main_iraq,
    finetune_labels=train_main_iraq_label
)
abortion_learning_trained_model.save('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/model/abortion_learning.h5')


firearm_learning_results, firearm_learning_trained_model, firearm_learning_history = train_and_evaluate(
    X_train_firearm, train_firearms_label_df,
    X_test_firearm, test_firearm_label_df,
    X_validate_main_iraq, val_main_iraq_label,
    X_test_main_iraq, test_main_iraq_label_df,
    model_party, early_stopping,
    model_name = "finetune_firearm",
    finetune_data=X_train_main_iraq,
    finetune_labels=train_main_iraq_label
)
firearm_learning_trained_model.save('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/model/firearm_learning.h5')

In [ ]:
merge_results, merge_trained_model, merge_history = train_and_evaluate(
    X_train_merge, concat_train_label_df,
    None, None,
    X_val_merge, concat_val_label_df,
    X_test_main_iraq, test_main_iraq_label_df,
    model_party, early_stopping, model_name = "merge_model"
)

merge_trained_model.save('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/model/merge_model.h5')


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import load_model

def evaluate_model(model, X_test, y_test, model_name, save_path=None):
    # Make predictions
    predictions = model.predict(X_test).flatten()
    predictions = (predictions > 0.5).astype(int)

    # Calculate metrics
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)

    if save_path:
        model.save(save_path)
        print(f'Model saved at {save_path}')

    # Print the results
    print(f'{model_name} Accuracy Score: {accuracy:.4f}')
    print(f'{model_name} Precision Score: {precision:.4f}')
    print(f'{model_name} Recall Score: {recall:.4f}')
    print(f'{model_name} F1 Score: {f1:.4f}')

    return accuracy, precision, recall, f1

vanilla_trained_model = load_model('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/model/vanilla_model.h5')
evaluate_model(model_party, X_test_main_iraq, test_main_iraq_label_df, vanilla_trained_model)

abortion_learning_trained_model = load_model('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/model/abortion_learning.h5')
evaluate_model(model_party, X_test_main_iraq, test_main_iraq_label_df, abortion_learning_trained_model)

abortion_no_finetune_trained_model = load_model('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/model/abortion_no_finetune.h5')
evaluate_model(model_party, X_test_main_iraq, test_main_iraq_label_df, abortion_no_finetune_trained_model)

firearm_learning_trained_model= load_model('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/model/firearm_learning.h5')
evaluate_model(model_party, X_test_main_iraq, test_main_iraq_label_df, firearm_learning_trained_model)

firearm_no_finetune_trained_model = load_model('/content/drive/MyDrive/CS_224N_Project/LSTM_Baseline/model/firearm_no_finetune.h5')
evaluate_model(model_party, X_test_main_iraq, test_main_iraq_label_df, firearm_no_finetune_trained_model)


66/66 [==============================] - 8s 95ms/step
<keras.src.engine.functional.Functional object at 0x7db774c2da50> Accuracy Score: 0.5797
<keras.src.engine.functional.Functional object at 0x7db774c2da50> Precision Score: 0.5597
<keras.src.engine.functional.Functional object at 0x7db774c2da50> Recall Score: 0.8219
<keras.src.engine.functional.Functional object at 0x7db774c2da50> F1 Score: 0.6659
66/66 [==============================] - 6s 86ms/step
<keras.src.engine.functional.Functional object at 0x7db774ac6ef0> Accuracy Score: 0.5797
<keras.src.engine.functional.Functional object at 0x7db774ac6ef0> Precision Score: 0.5597
<keras.src.engine.functional.Functional object at 0x7db774ac6ef0> Recall Score: 0.8219
<keras.src.engine.functional.Functional object at 0x7db774ac6ef0> F1 Score: 0.6659
66/66 [==============================] - 6s 84ms/step
<keras.src.engine.functional.Functional object at 0x7db76a5462c0> Accuracy Score: 0.5797
<keras.src.engine.functional.Functional object at 0

(0.579731027857829, 0.5596919127086007, 0.82186616399623, 0.6659030164184804)